# Full-120 timeline run with periodic shocks (stages on)

One combined scenario: **stages mode + periodic shocks**, processed as a
timeline. The 120 messages run in their `[Sent: ...]` order; the desk's
world clock moves to each message's sent date, queues drain with time
and grow with commits — the same mechanics as `harness/simulate.py`.

**Shock model** (`eval/shock.py`, mirrors the simulator): every other
week, one ACTIVE workshop is closed for 14 days. Targets come from a
seeded random stream, so the calendar is reproducible.

**Labels**: `data/f_heldout.json` (shock-free ground truth, untouched).
A shock-aware companion is derived below → `data/f_heldout_shock.json`:
each expectation is filtered by the closure active on its sent date.


In [ ]:
# ---- config ----
import sys, time, json, random
from pathlib import Path
from datetime import date, timedelta

ROOT = Path.cwd()
if (ROOT / 'desk').is_dir() is False and (ROOT.parent / 'desk').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

INBOX   = ROOT / 'data' / 'dispatch_requests_full120.txt'
LABELS  = ROOT / 'data' / 'f_heldout.json'
OVERLAY = ROOT / 'data' / 'f_heldout_shock.json'
CACHE   = ROOT / 'results' / 'full120'
CACHE.mkdir(parents=True, exist_ok=True)

DAY0      = date(2026, 1, 1)
STAGES    = True
PROVIDER  = 'deepseek'    # API switch
LIMIT     = 120           # set to 10 for a quick smoke
SEED      = 5105

# the API budget guards the HOSTED demo; raise it for offline eval runs
import os
os.environ.setdefault('API_CALLS_PER_HOUR', '2000')
os.environ.setdefault('API_CALLS_PER_DAY', '20000')

from desk import config
config.set_as_of(DAY0)
print('world starts', config.chat_today())


In [ ]:
# ---- inputs: inbox (with sent dates), labels, shock calendar ----
import re
from eval.run_eval import parse_inbox
from eval.shock import build_calendar, shock_on, DEFAULT_WINDOWS
from desk.register import WorkshopStore

SENT = re.compile(r'\[Sent: (\d{4}-\d{2}-\d{2})\]\s*$')

def sent_date(text):
    m = SENT.search(text)
    return date.fromisoformat(m.group(1)) if m else DAY0

requests = parse_inbox(INBOX)[:LIMIT]
requests.sort(key=lambda r: sent_date(r['text']))
labels = {l['request_id']: l for l in json.loads(LABELS.read_text(encoding='utf-8'))}

ws = WorkshopStore(stages=STAGES)
active = [w.workshop_id for w in ws.workshops.values() if w.status == 'ACTIVE']
CALENDAR = build_calendar(active, seed=SEED)
print('shock calendar:')
for c in CALENDAR:
    print(f"  day {c['start']:>3}-{c['end']:<3}  ({DAY0 + timedelta(days=c['start'])}"
          f" – {DAY0 + timedelta(days=c['end'])})  closes {c['target']}")


## Timeline runner

Per message, in sent-date order: world clock := sent date; queues
drained by elapsed days; active closure applied; `desk.handle_message`;
a recommended allocation is committed the way the simulator would
(start at max(day, queue_free), defect-priced work, closure pauses).


In [ ]:
from desk.desk import DispatchDesk
from desk.contracts import Behaviour, to_dict

class TimelineState:
    """Stand-in for desk.state: the tools read desk.state.queues,
    the timeline owns the values."""
    def __init__(self):
        self.queues = {}

desk = DispatchDesk(stages=STAGES, provider=PROVIDER)
# COUNTERFACTUAL (simulate.py, Assumption 1): the F-series orders are
# COMPLETE in the register — they finished in-house in reality. The
# replay asks 'what if they had been outsourced on their sent date',
# so the desk must see them as live work, or the agent correctly
# refuses to dispatch completed orders.
for _rec in desk.orders.orders:
    _rec['status'] = 'IN_PROGRESS'
    _rec['completed_date'] = None
    _rec['days_late'] = 0
    if _rec.get('current_stage') == 'COMPLETE':
        _rec['current_stage'] = None   # in-house finish is off-stage
desk.state = TimelineState()
queue_free = {wid: w.queue0 for wid, w in desk.workshops.workshops.items()}

OUT = CACHE / f'timeline_shock_stages-{int(STAGES)}@{PROVIDER}.jsonl'
done = {}
if OUT.exists():
    for line in OUT.read_text(encoding='utf-8').splitlines():
        if line.strip():
            row = json.loads(line)
            if 'request_id' in row and 'error' not in row:  # errors retry
                done[row['request_id']] = row

with OUT.open('a', encoding='utf-8') as f:
    for i, req in enumerate(requests, 1):
        if req['request_id'] in done:
            continue
        sent = sent_date(req['text'])
        day = (sent - DAY0).days
        config.set_as_of(sent)
        desk.state.queues = {w: max(0.0, queue_free[w] - day)
                             for w in queue_free}
        closure = shock_on(day, CALENDAR)
        for w in desk.workshops.workshops.values():
            if w.status in ('ACTIVE', 'SUSPENDED'):
                w.status = 'ACTIVE'
        if closure:
            desk.workshops.workshops[closure['target']].status = 'SUSPENDED'

        desk.agent.new_conversation()
        t0 = time.perf_counter()
        try:
            decision = desk.handle_message(req['text'], requester=req['speaker'],
                                           request_id=req['request_id'])
            record = {'request_id': req['request_id'],
                      'speaker': req['speaker'], 'raw': req['text'],
                      'sent_date': sent.isoformat(),
                      'shock_target': closure['target'] if closure else None,
                      'latency_s': round(time.perf_counter() - t0, 2),
                      **to_dict(decision)}
        except Exception as exc:
            record = {'request_id': req['request_id'],
                      'speaker': req['speaker'], 'raw': req['text'],
                      'sent_date': sent.isoformat(),
                      'shock_target': closure['target'] if closure else None,
                      'error': f'AGENT ERROR: {exc}'}
            decision = None

        if (decision is not None and decision.behaviour is Behaviour.ALLOCATE
                and decision.chosen is not None and decision.parsed.pieces):
            wid = decision.chosen.workshop_id
            w = desk.workshops.workshops[wid]
            start = max(day, queue_free[wid])
            work = int(decision.parsed.pieces) / w.capacity * (1 + 0.5 * w.defect_rate)
            finish = start + work
            if closure and closure['target'] == wid:
                if closure['start'] <= start < closure['end']:
                    finish = closure['end'] + work
                elif start < closure['start'] < finish:
                    finish += closure['end'] - closure['start']
            queue_free[wid] = finish
            record['committed'] = {'workshop_id': wid,
                                   'queue_free_after': round(finish, 2)}

        if 'error' not in record:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
            f.flush()
        print(f"[{i}/{len(requests)}] {req['request_id']} d{day:<3} "
              f"{'SHOCK:' + closure['target'] if closure else '-':<9} "
              f"{record.get('behaviour', record.get('error', '?'))}")


## Shock-aware labels → `data/f_heldout_shock.json`

The original labels stay untouched. The overlay records, per request:
the closure active on its sent date, the expectation with the closed
workshop removed (`expected_under_shock`), and an `affected` flag.


In [ ]:
from eval.shock import build_overlay, write_overlay

sent_by_id = {r['request_id']: sent_date(r['text']).isoformat()
              for r in requests}
overlay = build_overlay(list(labels.values()), sent_by_id, CALENDAR, DAY0)
write_overlay(LABELS, sent_by_id, CALENDAR, OVERLAY, DAY0)
affected = [o for o in overlay if o['affected']]
print(f"overlay written: {OVERLAY.name} — {len(affected)} affected of {len(overlay)}")
for o in affected[:10]:
    print(f"  {o['request_id']}: expected {o.get('workshop')} -> "
          f"{o['expected_under_shock']} (closed {o['shock_target']})")


## Scoring — original labels vs shock-corrected

Three numbers per run: behaviour accuracy (labels hold — the routing
code already sees the suspension through the tools), workshop accuracy
against the original expectations, and the **corrected** workshop score
where affected rows are judged against `expected_under_shock`.


In [ ]:
from eval.score_run import score_run, load_records

rows, summary = score_run(LABELS, OUT)
# labels name workshops by NAME ('Little Loom'), decisions carry ids
# ('W4') — resolve before comparing, or every workshop score is 0
from desk.register import WorkshopStore
_ws = WorkshopStore(stages=STAGES)
def _ids(names):
    return {_ws.resolve(w) or w for w in (names or [])}
for _l in labels.values():
    _l['workshop'] = sorted(_ids(_l.get('workshop')))
for _o in overlay:
    _o['workshop'] = sorted(_ids(_o.get('workshop')))
    _o['expected_under_shock'] = sorted(_ids(_o.get('expected_under_shock')))
records = load_records(OUT)
ov = {o['request_id']: o for o in overlay}

def actual_w(row):
    if row.get('chosen'):
        return row['chosen']['workshop_id']
    c = row.get('candidates') or []
    return c[0]['workshop_id'] if c else None

def corrected_workshop_ok(rec):
    o = ov.get(rec['request_id'])
    if not o or not o.get('workshop'):
        return None                      # no workshop label: not scored
    expected = o['expected_under_shock'] if o['affected'] else o['workshop']
    if not expected:
        return None                      # expectation emptied: manual review
    return actual_w(rec) in expected

behav = [r['behaviour_ok'] for r in rows if not r.get('error')]
w_orig = [r['workshop_ok'] for r in rows if r.get('workshop_ok') is not None]
w_corr = [x for x in (corrected_workshop_ok(r) for r in rows)
          if x is not None]
print(f"behaviour accuracy           : {sum(behav)}/{len(behav)} = {sum(behav)/len(behav):.3f}")
print(f"workshop accuracy (original) : {sum(w_orig)}/{len(w_orig)} = {sum(w_orig)/len(w_orig):.3f}")
print(f"workshop accuracy (corrected): {sum(w_corr)}/{len(w_corr)} = {sum(w_corr)/len(w_corr):.3f}")

manual = [o for o in affected if not o['expected_under_shock']]
print(f"\nmanual-review rows (expectation emptied by the closure): {len(manual)}")
for o in manual:
    print(f"  {o['request_id']}: label {o['behaviour']} {o.get('workshop')} "
          f"closed={o['shock_target']} on {o['sent_date']}")


## Reading the result

- **Behaviour accuracy** should stay close to the static runs: the
  router learns about closures through the tools, not the labels.
- **Corrected workshop accuracy** is the fair number under shock: the
  desk is right to skip a closed workshop, and the overlay says so.
- **Manual-review rows** are expectations the closure emptied — the
  ground truth itself is shock-dependent there; check by hand that the
  desk offered the next-best option (or refused / flagged impossible).
